In [ ]:
import pandas as pd
import numpy as np
import os

POP_SRC  = "/home/data/youngwoong/ST-GNN_Dataset/Data_Preprocessed/Land Use Regression/생활인구데이터/생활인구데이터.parquet"
SAVE_DIR = "/home/data/youngwoong/ST-GNN_Dataset/Data_Preprocessed/Land Use Regression/격자 기본"

df = pd.read_parquet(POP_SRC)
print(f'parquet shape: {df.shape}')
print(f'columns: {df.columns.tolist()}')
df.head(3)

In [ ]:
# ── 기준 타임스텝·격자 로드 ───────────────────────────────────────────────
timestamps_all = np.load(os.path.join(SAVE_DIR, 'timestamps_all.npy'))  # (18287,)
grid_basic     = pd.read_csv(os.path.join(SAVE_DIR, '격자_250m_4326.csv'))  # (10125, 7)

ts_dt  = pd.to_datetime(timestamps_all)          # DatetimeIndex, 18287개
T, G   = len(ts_dt), len(grid_basic)

print(f'timestamps_all : {T}개  ({ts_dt[0]} ~ {ts_dt[-1]})')
print(f'grid_basic     : {G}개 격자')

In [ ]:
# ── datetime 컬럼 생성 및 커버리지 확인 ───────────────────────────────────
df['datetime'] = (pd.to_datetime(df['일자'].astype(str), format='%Y%m%d')
                  + pd.to_timedelta(df['시간'], unit='h'))

pop_cells  = set(df['CELL_ID'].unique())
grid_cells = set(grid_basic['CELL_ID'].unique())
ts_set     = set(ts_dt)
pop_ts_set = set(df['datetime'].unique())

print('=== 격자 커버리지 ===')
print(f'  생활인구 격자   : {len(pop_cells):,}')
print(f'  grid_basic 격자 : {len(grid_cells):,}')
print(f'  교집합          : {len(pop_cells & grid_cells):,}  ← 매핑 대상')
print(f'  grid에만 존재   : {len(grid_cells - pop_cells):,}  → 0 으로 채움')
print(f'  parquet에만 존재: {len(pop_cells - grid_cells):,}  → 무시')
print()
print('=== 타임스텝 커버리지 ===')
print(f'  timestamps_all  : {T:,}')
print(f'  생활인구 타임스텝: {len(pop_ts_set):,}')
print(f'  교집합           : {len(ts_set & pop_ts_set):,}  ← 사용 타임스텝')
print(f'  생활인구에만 있음: {len(pop_ts_set - ts_set):,}  → 제외 (2023-10-01 00:00)')

In [ ]:
# ── (T, G) 배열 생성 ─────────────────────────────────────────────────────
# timestamps_all 순서 → t 인덱스 dict
ts_map   = {ts: i for i, ts in enumerate(ts_dt)}
# grid_basic fid 순서 (fid 1-based → 0-based 인덱스) → g 인덱스 dict
cell_map = {row['CELL_ID']: row['fid'] - 1
            for _, row in grid_basic[['CELL_ID', 'fid']].iterrows()}

pop_arr = np.zeros((T, G), dtype=np.float32)

# timestamps_all에 없는 타임스텝 및 grid_basic에 없는 격자 동시 필터
df_valid = df[df['datetime'].isin(ts_map) & df['CELL_ID'].isin(cell_map)].copy()
print(f'필터 후 행 수: {len(df_valid):,}  (전체 {len(df):,}에서 {len(df)-len(df_valid):,}개 제외)')

t_idx = df_valid['datetime'].map(ts_map).values
g_idx = df_valid['CELL_ID'].map(cell_map).values
pop_arr[t_idx, g_idx] = df_valid['생활인구합계'].values

print(f'\npop_arr shape : {pop_arr.shape}')
print(f'  전체 셀 중 0인 비율 : {(pop_arr == 0).mean():.2%}')
print(f'  생활인구 통계:')
nonzero = pop_arr[pop_arr > 0]
print(f'    min={nonzero.min():.1f}  median={np.median(nonzero):.1f}  '
      f'mean={nonzero.mean():.1f}  max={nonzero.max():.1f}')

In [ ]:
# ── 저장 ─────────────────────────────────────────────────────────────────
out_path = os.path.join(SAVE_DIR, 'population_hourly.npy')
np.save(out_path, pop_arr)
print(f'저장 완료 → {out_path}')
print(f'  shape  : {pop_arr.shape}   (T=18287, G=10125)')
print(f'  dtype  : {pop_arr.dtype}')
print(f'  size   : {pop_arr.nbytes / 1024**2:.1f} MB')

In [ ]:
# ── 검증 ─────────────────────────────────────────────────────────────────
loaded = np.load(out_path)
assert loaded.shape == (T, G), f'shape 불일치: {loaded.shape}'

# ndvi_hourly와 shape 동일 확인
ndvi = np.load(os.path.join(SAVE_DIR, 'ndvi_hourly.npy'))
assert loaded.shape == ndvi.shape, f'ndvi shape와 다름: {ndvi.shape}'
print(f'✓ shape 검증 통과: {loaded.shape}  (ndvi_hourly와 동일)')

# 샘플 격자 시계열 확인
sample_cell = '다사35005075'
if sample_cell in cell_map:
    g = cell_map[sample_cell]
    ts_sample = ts_dt[:24]
    pop_sample = loaded[:24, g]
    print(f'\n샘플 격자 {sample_cell} (g={g}) 첫 24시간 생활인구:')
    for t, p in zip(ts_sample, pop_sample):
        print(f'  {t}  {p:.2f}')

In [ ]:
# ── 시간대별 평균 생활인구 (전체 격자) 시각화 ────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

hourly_mean = loaded.mean(axis=1)   # (T,) — 타임스텝별 전체 격자 평균
hour_of_day = ts_dt.hour

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# 전체 시계열 (1000개 샘플링)
ax = axes[0]
step = max(1, T // 1000)
ax.plot(ts_dt[::step], hourly_mean[::step], linewidth=0.6, color='#3498db')
ax.set_title('생활인구 시간 평균 (전체 격자)', fontsize=12)
ax.set_xlabel('시간')
ax.set_ylabel('평균 생활인구합계')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
plt.setp(ax.xaxis.get_majorticklabels(), rotation=30)
ax.grid(alpha=0.3)

# 시간대별 평균 패턴
ax = axes[1]
hourly_pattern = pd.Series(hourly_mean, index=ts_dt).groupby(ts_dt.hour).mean()
ax.bar(hourly_pattern.index, hourly_pattern.values, color='#e67e22', alpha=0.85, edgecolor='white')
ax.set_title('시간대별 평균 생활인구 패턴', fontsize=12)
ax.set_xlabel('시간 (hour)')
ax.set_ylabel('평균 생활인구합계')
ax.set_xticks(range(0, 24))
ax.grid(axis='y', alpha=0.3)

plt.suptitle('population_hourly.npy 검증', fontsize=13)
plt.tight_layout()
plt.show()

print('\n=== 최종 요약 ===')
print(f'  파일      : {out_path}')
print(f'  shape     : {loaded.shape}  dtype={loaded.dtype}')
print(f'  기간      : {ts_dt[0]} ~ {ts_dt[-1]}')
print(f'  격자 수   : {G}  (생활인구 있는 격자: {(loaded.sum(axis=0) > 0).sum():,})')
print(f'  사용 방법 : ndvi_hourly.npy 와 동일하게 population_all[target_global_idx][:, sta_to_grid]')